# MentorHub Colab Dev Parser

仅用于本地开发测试：本地 MentorHub 通过 Google Drive 同步目录提交解析任务，Colab 负责 MinerU / 图片解析，然后把结果写回 Drive。

**使用方式：** 选择 GPU Runtime（可选）→ `Runtime > Run all` → 授权 Google Drive。之后保持此 Notebook 运行即可。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 必须与本地 Google Drive for desktop 中的 MentorHub-Colab 文件夹对应。
QUEUE_ROOT = '/content/drive/MyDrive/MentorHub-Colab'
print('Queue root:', QUEUE_ROOT)


In [ ]:
import subprocess, sys

# 与当前本地开发环境保持同一 MinerU 主版本。
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', 'mineru==4.0.2'
])
print('MinerU installed')


In [ ]:
import shutil, subprocess

nvidia_smi = shutil.which('nvidia-smi')
if not nvidia_smi:
    raise RuntimeError('未检测到 GPU。请先在 运行时 > 更改运行时类型 中选择 GPU，然后重新全部运行。')

print('GPU info:')
subprocess.run([nvidia_smi, '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)


In [ ]:
from pathlib import Path
import importlib.util, time

worker = Path(QUEUE_ROOT) / 'worker' / 'colab_queue_worker.py'
print('Checking MentorHub worker script...')
while not worker.exists():
    print('  worker not found yet:', worker)
    time.sleep(3)

pending_dir = Path(QUEUE_ROOT) / 'pending'
pending_jobs = sorted(p.name for p in pending_dir.glob('*.zip'))
print('Worker ready:', worker)
print('Pending jobs:', pending_jobs)

spec = importlib.util.spec_from_file_location('mentorhub_colab_queue_worker', worker)
worker_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(worker_module)
worker_module.run(Path(QUEUE_ROOT), poll_seconds=2, once=False)
